# Two faces, one acoustic drive: the refinement test
### Executed research companion · 23 September 2026

**Result: the apparent 10 nm design-grid pass does not survive surface refinement.**
This notebook reconstructs saved stationary solutions for the same 448 coherent
source commands at 7.2 MHz. It does not optimize, rerun the wave solver, or manufacture a formation trajectory.

The requirement is maximum **cycle-mean height error ≤ 10 nm on each 2 mm-radius clear aperture**, without piston/tilt removal.
The modeled chamber is a sealed three-fluid cylinder, 4 mm in radius and 6 mm high.
Materials are hypothetical; ports are idealized full-azimuth rings/bands, not 448 fabricated point emitters.
The model is axisymmetric, inviscid in its harmonic field and isothermal. Three-dimensional stability, streaming, heat and experimental validation are absent.

**Reading route:** evidence table → meridional surfaces → pupil errors → convergence → sources → physical-time pressure animation → reproducibility audit.
[Scientific interpretation](../docs/precision-results.md) · [Material feasibility](../docs/material-feasibility.md) · [Theory](../artifacts/theory/acoustic-fluid-shaping-theory.pdf)

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/acoustic_freeform").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from acoustic_freeform.apparatus.config import DualConfig
from acoustic_freeform.mechanics.surface import DualSurface, CartesianPatch
from acoustic_freeform.core.provenance import capture_execution

OUT = ROOT / "artifacts/studies/S02-independent-two-face/notebooks/04_dual_cartesian"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": .18})
COLORS = ["#2166ac", "#d6604d", "#4d9221"]
CASE = "asymmetric-pair-A"
BASE = ROOT / "artifacts/studies/S02-independent-two-face/dual-precision-2026-09-23"
specs = [("52 elements", "verify-mean-448-7p2mhz-retry1", "design"),
         ("80 elements", "verify-mean-448-surface-refinement", "design"),
         ("104 elements", "verify-mean-448-surface-refinement", "refined-1")]
runs, inputs = [], {}
def read_json(p):
    inputs[str(p.relative_to(ROOT))] = hashlib.sha256(p.read_bytes()).hexdigest()
    return json.loads(p.read_text())
for label, folder, grid in specs:
    directory = BASE / folder
    config = read_json(directory / "config.json")
    records = read_json(directory / "results.json")
    case = next(c for c in records if c["case"] == CASE)
    record = next(g for g in case["grids"] if g["grid"] == grid)
    path = directory / CASE / f"{grid}-state.npz"
    inputs[str(path.relative_to(ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
    with np.load(path, allow_pickle=False) as data:
        state = {k: data[k].copy() for k in data.files}
    cfg = DualConfig(**record["numerics"])
    space = DualSurface(cfg)
    target_inputs = next(c for c in config["cases"] if c["name"] == CASE)["faces"]
    targets = [CartesianPatch(cfg, j, f["optical"], f["vertex_displacement_m"])
               for j, f in enumerate(target_inputs)]
    runs.append(dict(label=label, record=record, state=state, cfg=cfg,
                     space=space, targets=targets))
for run in runs[1:]:
    np.testing.assert_array_equal(run["state"]["source_velocity_m_s"],
                                  runs[0]["state"]["source_velocity_m_s"])
    for key, value in runs[0]["record"]["numerics"].items():
        if key != "surface_elements":
            assert run["record"]["numerics"][key] == value, key
assert all(r["record"]["declared_accuracy_target"] == "cycle_mean" for r in runs)
print("Loaded 3 saved equilibria. Commands and all settings except surface resolution match exactly.")
def savefig(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=180, bbox_inches="tight")
    fig.savefig(OUT / f"{name}.svg", bbox_inches="tight")
    plt.show()

## 1. What actually passed?

The acceptance values below are the saved independent radial maximum searches,
not the residual minimized by the source inverse. A tiny force-balance residual
does not imply a small target error. The 52-element solution was accepted at its
initial seed by a fresh force-balance evaluation; it is not a computed formation process.

In [ ]:
rows = ["| Surface elements | Front max (nm) | Back max (nm) | Mean criterion | Root residual (m) | Acoustic work (W) | Sampled peak (MPa) |",
        "|---:|---:|---:|:---:|---:|---:|---:|"]
for run in runs:
    rec = run["record"]
    e = [f["max_error_m"] * 1e9 for f in rec["faces"]]
    rows.append(f'| {run["cfg"].surface_elements} | {e[0]:.6g} | {e[1]:.6g} | '
                f'{"PASS on this grid" if max(e) <= 10 else "FAIL"} | '
                f'{rec["equilibrium"]["compliance_residual_max_m"]:.3e} | '
                f'{rec["acoustics"]["source_power_w"]:.4f} | '
                f'{rec["acoustics"]["sampled_pressure_peak_pa"] / 1e6:.3f} |')
display(Markdown("\n".join(rows)))
display(Markdown("**No mesh-converged or physical 10 nm result is established.** "
                 "The acoustic mesh is unchanged in this comparison."))

## 2. Both surfaces in the actual chamber

The cross-section uses laboratory coordinates and equal geometric scale.
The non-optical outer annuli enforce volume and rim constraints; they are not
part of the Cartesian clear apertures. The three-dimensional shape is the
surface of revolution of each meridional curve. Only the declared cylinder
and its two liquid interfaces are drawn.

In [ ]:
rr = np.linspace(0, runs[0]["cfg"].radius_m, 4001)
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
cfg = runs[0]["cfg"]
for x in (-cfg.radius_m, cfg.radius_m):
    ax.plot([x * 1e3] * 2, np.array(cfg.levels_m)[[0, -1]] * 1e3, color=".25", lw=2)
for z in (cfg.levels_m[0], cfg.levels_m[-1]):
    ax.plot([-cfg.radius_m * 1e3, cfg.radius_m * 1e3], [z * 1e3] * 2, color=".25", lw=2)
signed_r = np.r_[-rr[:0:-1], rr]
for j in range(2):
    target = cfg.levels_m[j+1] + runs[0]["targets"][j].evaluate(abs(signed_r))
    ax.plot(signed_r * 1e3, target * 1e3, "k--", lw=1.3,
            label="Requested geometry" if j == 0 else None)
    for color, run in zip(COLORS, runs):
        z = cfg.levels_m[j+1] + run["space"].evaluate(run["state"]["coefficients_m"][j], abs(signed_r))
        ax.plot(signed_r * 1e3, z * 1e3, color=color, lw=1.2,
                label=run["label"] if j == 0 else None)
for x in (-cfg.clear_radius_m, cfg.clear_radius_m):
    ax.axvline(x * 1e3, ls=":", color=".55")
ax.text(0, 2.1, "Upper fluid", ha="center", color=".4")
ax.text(0, 0, "Lens fluid", ha="center", color=".4")
ax.text(0, -2.1, "Lower fluid", ha="center", color=".4")
ax.set(xlabel="Signed radial coordinate (mm)", ylabel="Laboratory z (mm)",
       title="Stationary two-face geometry · no geometric exaggeration", aspect="equal")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
savefig(fig, "01_chamber")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for j, ax in enumerate(axes):
    ax.plot(rr * 1e3, runs[0]["targets"][j].evaluate(rr) * 1e6, "k--", label="Target")
    for color, run in zip(COLORS, runs):
        ax.plot(rr * 1e3, run["space"].evaluate(run["state"]["coefficients_m"][j], rr) * 1e6,
                color=color, label=run["label"])
    ax.axvspan(cfg.clear_radius_m * 1e3, cfg.radius_m * 1e3, color=".9", label="Non-optical annulus")
    ax.set(title=["Front / lower interface", "Back / upper interface"][j],
           xlabel="Radius (mm)", ylabel="Displacement from flat interface (µm)")
axes[1].legend(fontsize=8)
fig.suptitle("Surface profiles · axes use different units; not an equal-scale geometry")
savefig(fig, "02_profiles")

## 3. Aperture error, not visual similarity

The target is evaluated from its Cartesian definition. These plots use 8,001
radii independent of the acoustic integration grid. Recorded maxima also use
local extremum searches; neither is a rigorous interval bound. The shaded
±10 nm band is the acceptance requirement.

In [ ]:
rp = np.linspace(0, cfg.clear_radius_m, 8001)
profiles = np.array([[r["space"].evaluate(r["state"]["coefficients_m"][j], rp)
                      - r["targets"][j].evaluate(rp) for j in range(2)] for r in runs])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for j, ax in enumerate(axes):
    ax.axhspan(-10, 10, color="#b8e0d2", alpha=.55, label="±10 nm requirement")
    for color, run, err in zip(COLORS, runs, profiles):
        ax.plot(rp * 1e3, err[j] * 1e9, color=color, label=run["label"])
    ax.set_yscale("symlog", linthresh=10)
    ax.set(xlabel="Pupil radius (mm)", ylabel="Signed mean-height error (nm)",
           title=["Front", "Back"][j])
axes[1].legend(fontsize=9)
fig.suptitle("Fixed-command errors · symmetric-log scale, linear between −10 and +10 nm")
savefig(fig, "03_pupil_errors")
recorded = np.array([[f["max_error_m"] for f in r["record"]["faces"]] for r in runs])
sampled = np.max(abs(profiles), axis=2)
# Independent grid is a cross-check of saved values, not a new physical certificate.
assert np.all(sampled <= recorded + 2e-12)
np.testing.assert_allclose(sampled, recorded, rtol=2e-3, atol=2e-12)
print("Independent dense-grid maxima agree with saved searches within declared numerical tolerances.")

## 4. Refinement failure and separate acoustic diagnostics

All three curves use exactly the same commands, frequency and acoustic mesh.
The x-axis below is **surface discretization**, not time. There is no justified
convergence rate here, and increasing emitters would not repair this validation
failure by itself. Carrier displacement is diagnostic only for the selected
cycle-mean requirement.

In [ ]:
elements = [r["cfg"].surface_elements for r in runs]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for j, name in enumerate(["Front", "Back"]):
    axes[0].semilogy(elements, recorded[:, j] * 1e9, "o-", label=name)
axes[0].axhline(10, color="k", ls="--", label="10 nm")
axes[0].set(xlabel="Surface elements", ylabel="Maximum mean error (nm)", title="Target accuracy")
axes[0].legend(fontsize=8)
residuals = [r["record"]["equilibrium"]["compliance_residual_max_m"] for r in runs]
axes[1].semilogy(elements, residuals, "o-", color="#762a83")
axes[1].set(xlabel="Surface elements", ylabel="Compliance residual (m)", title="Force balance ≠ target accuracy")
carriers = np.array([r["record"]["acoustics"]["pupil_sampled_carrier_height_peak_m"] for r in runs])
for j, name in enumerate(["Front", "Back"]):
    axes[2].plot(elements, carriers[:, j] * 1e9, "o-", label=name)
axes[2].set(xlabel="Surface elements", ylabel="Sampled carrier amplitude (nm)", title="Separate ultrasonic motion")
axes[2].legend()
savefig(fig, "04_refinement")

## 5. The actual shared commands

These are peak complex source velocities under the convention
$\mathrm{Re}\{g\exp(-i\omega t)\}$.
Index order is the archived source order; this is not a spatial transducer map.
The ideal ports occupy full-azimuth bottom/top annuli and sidewall bands.
Acoustic source work is not an electrical-power calibration.

In [ ]:
drive = runs[-1]["state"]["source_velocity_m_s"]
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, constrained_layout=True)
axes[0].plot(np.arange(len(drive)), abs(drive), ".", ms=3, color="#2166ac")
axes[0].axhline(cfg.max_source_speed_m_s, color="k", ls="--", label="Imposed component bound")
axes[0].set(ylabel="Peak speed (m/s)", title=f"One unchanged {len(drive)}-component drive")
axes[0].legend()
axes[1].plot(np.arange(len(drive)), np.angle(drive), ".", ms=3, color="#d6604d")
axes[1].set(xlabel="Archived source index", ylabel="Phase (rad)", ylim=(-np.pi, np.pi))
savefig(fig, "05_commands")

## 6. Physical-time animation: acoustic pressure, not surface formation

The final 104-element equilibrium includes a solved complex pressure field.
We animate the pressure at saved **on-axis pressure nodes**, using
$p(z,t)=\mathrm{Re}\{P(z)e^{-i2\pi f t}\}$ with $f=7.2$ MHz.
One cycle is approximately 138.89 ns. Playback is deliberately slowed down;
the title gives physical time, not screen time.

The pressure-line geometry changes with physical time. The dashed interface
positions remain at their cycle-mean locations: **this is not a moving-surface,
formation, streaming, or stability simulation**. Lines connect saved nodal
values for display; the plot is not an exact finite-element interpolation
between nodes. No desired-shape interpolation is used.

In [ ]:
final = runs[-1]
nodes = final["state"]["pressure_nodes_m"]
pressure = final["state"]["pressure_pa"]
axis = np.flatnonzero(np.isclose(nodes[0], 0, rtol=0, atol=1e-13))
axis = axis[np.argsort(nodes[1, axis])]
z_axis, p_axis = nodes[1, axis], pressure[axis]
assert len(axis) > 20 and np.all(np.isfinite(p_axis))
assert np.all(np.diff(z_axis) > 0)
frequency = final["cfg"].frequency_hz
period_s = 1 / frequency
times_s = np.linspace(0, period_s, 49)
frames_pa = np.real(p_axis[None, :] * np.exp(-2j * np.pi * frequency * times_s[:, None]))
np.testing.assert_allclose(frames_pa[0], frames_pa[-1], atol=1e-6)
assert np.max(abs(frames_pa[0] - frames_pa[12])) > 1
np.savez_compressed(OUT / "axis-pressure-cycle.npz", z_m=z_axis, time_s=times_s,
                    pressure_pa=frames_pa, pressure_phasor_pa=p_axis)
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
line, = ax.plot(z_axis * 1e3, frames_pa[0] / 1e6, color="#2166ac", lw=1.4)
limit = float(np.max(abs(p_axis))) / 1e6 * 1.1
ax.set(xlabel="On-axis laboratory z (mm)", ylabel="Instantaneous acoustic pressure (MPa)",
       ylim=(-limit, limit))
for j in range(2):
    z_interface = cfg.levels_m[j+1] + final["space"].evaluate(
        final["state"]["coefficients_m"][j], np.array([0.]))[0]
    ax.axvline(z_interface * 1e3, color=".4", ls="--", lw=1)
title = ax.set_title("")
def update(k):
    line.set_ydata(frames_pa[k] / 1e6)
    title.set_text(f"Acoustic pressure only · physical t = {times_s[k] * 1e9:.2f} ns"
                   "\nStationary mean interfaces · playback slowed; not formation")
    return line, title
animation = FuncAnimation(fig, update, frames=len(times_s), interval=90, blit=False)
animation_html = animation.to_jshtml(default_mode="loop")
# Text controls remain readable without the external icon stylesheet.
animation_html = re.sub(r'<link[^>]*>', '', animation_html)
animation_html = re.sub(r'(<button[^>]*title="([^"]+)"[^>]*>)\s*<i[^>]*></i>', r'\1\2', animation_html)
animation_html += '<style>.anim-buttons button {width:auto; padding:5px; margin:2px}</style>'
plt.close(fig)
(OUT / "pressure-cycle.html").write_text(
    '<!doctype html><meta charset="utf-8"><title>Acoustic pressure cycle</title>'
    '<h1>Reconstructed harmonic pressure at a stationary equilibrium</h1>'
    '<p>Physical time is in nanoseconds. Playback is slowed. Dashed lines mark '
    'stationary mean interfaces. No surface formation is simulated.</p>' + animation_html)
display(HTML(animation_html))

## 7. Reproducibility and limitations

The notebook reads completed artifacts without changing them. The following
audit records input hashes, the analysis source and its execution environment.
All data files use SI units; conversions occur only in plots/tables.

**Numerical checks here:** exact command identity; unchanged acoustic settings;
dense independent pupil evaluation versus recorded maxima; finite periodic
pressure reconstruction with nonconstant frames. These do not rerun the PDE,
validate a physical material, prove a rigorous maximum bound, or establish
three-dimensional stability.

**Next scientific task:** resolve traction integration and surface/acoustic
convergence at fixed commands, then qualify real materials and physical drive
limits. The latest refined result misses the target on both faces.

In [ ]:
source = ROOT / "notebooks/04_dual_cartesian.ipynb"
inputs[str(source.relative_to(ROOT))] = hashlib.sha256(source.read_bytes()).hexdigest()
capture_execution(OUT)
(OUT / "config.json").write_text(json.dumps({
    "operation": "Read-only analysis of saved two-face stationary solutions",
    "accuracy_target": "cycle_mean", "input_sha256": inputs,
    "animation": "Harmonic pressure reconstruction; not surface formation",
    "pressure_cycle_samples": len(times_s), "pupil_plot_samples": len(rp)
}, indent=2) + "\n")
validation = {
    "same_commands_exactly": True, "only_surface_resolution_changed": True,
    "dense_error_crosscheck_passed": True,
    "pressure_periodicity_and_frame_change_passed": True,
    "axis_node_count": len(axis), "physical_period_s": period_s,
    "sampled_max_error_m": sampled.tolist(), "recorded_max_error_m": recorded.tolist(),
    "new_wave_solves_performed": False, "physical_accuracy_certified": False
}
(OUT / "validation.json").write_text(json.dumps(validation, indent=2) + "\n")
(OUT / "validation-report.md").write_text(
    "# Two-face notebook validation\n\n"
    "Executed read-only reconstruction of three saved stationary solutions. "
    "Input hashes and analysis provenance are retained in config.json and provenance/.\n\n"
    "Checks: identical commands and acoustic settings, independent pupil grid "
    "against saved maxima, finite pressure cycle with distinct frames and periodic endpoints.\n\n"
    "The 10 nm mean criterion fails on both refined faces. "
    "The animation reconstructs harmonic pressure over one physical acoustic period; "
    "it is not surface motion or formation. No new PDE solve or physical validation.\n")
display(Markdown("**Audit complete.** Figures, SI pressure-cycle data, standalone animation, "
                 "configuration, provenance and validation report are saved in "
                 "\`artifacts/studies/S02-independent-two-face/notebooks/04_dual_cartesian/\`."))
print(json.dumps(validation, indent=2))